# Dissipative Two-Site XX Chain

This notebook is a compact introduction to the many-body `lindblad` interface in `EDKit.jl`. We study the smallest interacting open system that still shows nontrivial physics: a two-site XX chain with a local loss channel on the first site.

The point is not large-scale numerics. The point is to see clearly how the API maps onto a physical Lindblad problem, what observables you can track, and what qualitative behavior to expect.

## Physical model

We evolve the density matrix under

$$\dot{\rho} = -i[H, \rho] + L \rho L^\dagger - \tfrac{1}{2}\{L^\dagger L, \rho\},$$

with

$$H = J(X_1 X_2 + Y_1 Y_2), \qquad L = \sqrt{\gamma} \, \sigma^-_1.$$

The Hamiltonian moves a spin excitation between the two sites, while the jump operator removes population from site 1. We start from `|10>` in the occupation language: site 1 occupied, site 2 empty.

In [ ]:
import Pkg

function find_repo_root(start = pwd())
    dir = abspath(start)
    while true
        if isfile(joinpath(dir, "Project.toml")) && isfile(joinpath(dir, "src", "EDKit.jl"))
            return dir
        end
        parent = dirname(dir)
        parent == dir && error("Could not locate EDKit.jl repo root from $(start)")
        dir = parent
    end
end

const DEV = true
const REPO_ROOT = find_repo_root()
Pkg.activate(REPO_ROOT)
if DEV
    pushfirst!(LOAD_PATH, REPO_ROOT)
end

using EDKit
using LinearAlgebra
using Statistics


In [ ]:
J = 0.7
gamma = 0.35
dt = 0.02
steps = 120
L = 2

h2 = spin((J, "XX"), (J, "YY"))
H = Array(operator(h2, [1, 2], L))
jump = sqrt(gamma) * Array(operator(ComplexF64[0 1; 0 0], [1], L))
lb = lindblad(H, [jump])

basis = TensorBasis(L = L, base = 2)
psi0 = productstate([0, 1], basis)
dm = densitymatrix(psi0)

n1_op = Hermitian(Array(operator([1.0 0.0; 0.0 0.0], [1], L)))
n2_op = Hermitian(Array(operator([1.0 0.0; 0.0 0.0], [2], L)))


## Time evolution helper

`EDKit.lindblad` returns a callable object. We repeatedly apply it for small time steps and record the observables of interest. For this tiny model the solver is already exact enough that the numerical output is easy to interpret directly.

In [ ]:
function evolve_chain(lb, dm0, n1_op, n2_op; dt, steps, order = 8)
    times = collect(0:steps) .* dt
    dm = dm0
    n1 = zeros(length(times))
    n2 = zeros(length(times))
    traces = zeros(length(times))
    entropies = zeros(length(times))

    n1[1] = expectation(n1_op, dm)
    n2[1] = expectation(n2_op, dm)
    traces[1] = real(tr(dm.ρ))
    entropies[1] = EDKit.entropy(dm)

    for step in 1:steps
        dm = lb(dm, dt, order = order)
        n1[step + 1] = expectation(n1_op, dm)
        n2[step + 1] = expectation(n2_op, dm)
        traces[step + 1] = real(tr(dm.ρ))
        entropies[step + 1] = EDKit.entropy(dm)
    end

    (; times, n1, n2, traces, entropies, final_density = Array(dm))
end

results = evolve_chain(lb, dm, n1_op, n2_op; dt = dt, steps = steps)


## What to look for

- `n1(t)` should decrease because the jump acts on site 1.
- `n2(t)` can rise transiently because the XX term moves the excitation before it is lost.
- `tr(ρ)` should stay at 1 to numerical precision.
- The von Neumann entropy can increase because the state becomes mixed under dissipation.

In [ ]:
summary = (
    final_n1 = results.n1[end],
    final_n2 = results.n2[end],
    max_total_occupation = maximum(results.n1 .+ results.n2),
    final_total_occupation = results.n1[end] + results.n2[end],
    max_trace_error = maximum(abs.(results.traces .- 1)),
    final_entropy = results.entropies[end],
)
summary


In [ ]:
try
    using Plots
    p1 = plot(results.times, results.n1; label = "n1", xlabel = "time", ylabel = "occupation")
    plot!(p1, results.times, results.n2; label = "n2")
    p2 = plot(results.times, results.n1 .+ results.n2; label = "n1 + n2", xlabel = "time", ylabel = "total occupation")
    p3 = plot(results.times, results.entropies; label = "entropy", xlabel = "time", ylabel = "S(ρ)")
    p4 = plot(results.times, results.traces; label = "trace", xlabel = "time", ylabel = "tr(ρ)")
    plot(p1, p2, p3, p4; layout = (2, 2), size = (900, 700))
catch err
    @info "Plots not available; returning arrays instead" error = err
    (
        occupation_site_1 = results.n1,
        occupation_site_2 = results.n2,
        total_occupation = results.n1 .+ results.n2,
        entropy = results.entropies,
        trace = results.traces,
    )
end


## Interpretation

This is the basic many-body Lindblad workflow in `EDKit.jl`: build dense operators with `operator`, wrap them in `lindblad`, and evolve a `DensityMatrix`. Even in this tiny example you can already see the competition between coherent hopping and local loss.

For larger systems you would still use the same interface, but you would usually work in a symmetry sector or switch to a more specialized formulation when the model is quadratic. The next notebook shows exactly that quadratic shortcut.